# Notebook 10 — Total Correctness + Worked Examples

**Covers Chapter 4 §4.6–4.9.** N9 gave us partial correctness — "if it terminates, the answer is right." This notebook adds the missing piece: **proving that the program *always* terminates**.

Then we walk through the four further worked examples from §4.7 (β, gcd, quadratic, Collatz), discuss practical guidance from §4.8, and tackle exercises 32–38.

## What you'll be able to do by the end

- State the total-correctness while rule and identify a **loop variant**.
- Adapt a partial-correctness derivation to a total-correctness one.
- Distinguish loop *invariants* from loop *variants* — different concepts, both needed for total correctness.
- Solve exercises 32–38 (specs + derivations for power, sum, integer log, divisibility test, repeat-until).

In [1]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks']:
    if (candidate / 'while_lang.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise ImportError('could not find while_lang.py')

from while_lang import (
    parse, run, trace, count_steps,
    verify_triple, report_triple,
    StepBudgetExceeded,
)
import math
print('imports OK')

imports OK


## §1. The total correctness system (§4.6)

We use the notation:

$$\{P\}\ S\ \{\Downarrow Q\}$$

**Read it as:** "If S starts in a state satisfying P, then S **terminates**, and the resulting state satisfies Q."

Five of the six rules are **identical** to the partial-correctness ones (assignment, skip, composition, if, consequence — just sprinkle `⇓` everywhere). The only rule that needs a real change is the **while** rule.

### The total while rule

$$\dfrac{P \to a \ge 0 \qquad \{P \,\land\, \mathcal{B}\,b \,\land\, \mathcal{A}\,a = k\}\ S\ \{\Downarrow P \,\land\, \mathcal{A}\,a < k\}}{\{P\}\ \textbf{while } b \textbf{ do } S\ \{\Downarrow P \,\land\, \neg\,\mathcal{B}\,b\}}$$

Compared to the partial version, **two new ingredients**:

1. **`P → a ≥ 0`** — the loop invariant guarantees the variant `a` is non-negative.
2. **`a` decreases each iteration** — represented by introducing a fresh ghost `k = a` in the precondition and showing `a < k` in the postcondition of the body.

**`a` is the loop variant.** It's a generalised arithmetic expression (any maths goes — same expressive freedom as predicates).

### Why this guarantees termination

Each body execution decreases `a` by at least 1 (the integers don't have arbitrary infinite-decreasing chains starting at non-negative). After at most some number of iterations, `a` would have to go below 0 — but the invariant forbids that. So the loop *must* exit, which means `b` must have become false.

**Loop variant != loop invariant.** Easy to confuse:

| Concept | What it is | What it does |
|:--|:--|:--|
| **Loop invariant** | A predicate `P` that's preserved by the body | Encodes what the loop is *computing* |
| **Loop variant** | A non-negative arithmetic expression `a` that strictly decreases each iteration | Proves the loop *terminates* |

Both are needed for total correctness. **Finding the variant is usually easier** — it's often something like "the loop counter" or "the bound minus the counter."

### Total division proof — minor adjustments

From §4.6.3, the partial proof of the division program upgrades to total correctness with one new fact: **`r` is a loop variant** for the body `d := d + 1; r := r − n`:

- `r ≥ 0` is part of the invariant. ✓
- Each body execution does `r := r − n`, where `n ≥ 1`, so `r` strictly decreases. ✓

That's it — same proof, plus the variant. Done.

## §2. The four worked examples from §4.7

Quick walkthrough of each. Full derivations in the chapter; here we hit the highlights and verify empirically.

### §4.7.1 The β function — `if` without loops

```
if 0 ≤ n then x := 2 × n
else x := −2 × n − 1
```

The chapter shows `{ } S { ⇓ x ∈ ℕ }`. We extended this in N9 (Exercise 31) to `{n = n̄} S {⇓ x = β(n̄)}`.

**No loops, so no variants needed** — total correctness is automatic if the partial-correctness derivation goes through. The if rule for both versions is identical; just keep the `⇓` arrow.

### §4.7.2 Greatest common divisor

```
x := m; y := n;
while ¬(x = y) do (
    if y < x then x := x − y else y := y − x
)
```

**Loop invariant:** `R = ∀i ∈ ℕ. (i divides m̄, n̄) ↔ (i divides x and y)`.

**Why this works:** if `y < x`, then `x := x − y`. Any `i` that divides `x` and `y` also divides `x − y`. Any `i` that divides `x − y` and `y` also divides `x = (x − y) + y`. So the set of common divisors of `x, y` doesn't change. Same in the symmetric case.

**At loop exit (`x = y`):** `i` divides `m̄, n̄` ↔ `i` divides `x = y`. So the GCD of `m̄, n̄` equals `x = y` — exactly the answer.

**Loop variant for total correctness:** `x + y`.

- Always non-negative (we ensure `x, y ∈ ℕ \ {0}`).
- Each iteration decreases either `x` or `y` strictly, so `x + y` strictly decreases.

**Why not just `x`?** Because in the `else` branch (`y := y − x`) `x` doesn't change. Same for `y` alone. Their sum *does* always decrease. **Lesson:** sometimes you need to combine variables to get a working variant.

In [2]:
gcd_prog = '''
    x := m; y := n;
    while !(x = y) do (
        if !(x <= y) then
            x := x - y
        else (
            y := y - x
        )
    )
'''

# Verify {m, n ∈ ℕ \ {0}} S {⇓ x = y = gcd(m̄, n̄)}
ok = 0
for m_in in range(1, 25):
    for n_in in range(1, 25):
        res = verify_triple(
            precond  = lambda s, m=m_in, n=n_in: s.get('m', 0) == m and s.get('n', 0) == n,
            prog     = gcd_prog,
            postcond = lambda s, m=m_in, n=n_in: s.get('x', 0) == math.gcd(m, n) and s.get('y', 0) == math.gcd(m, n),
            sample_states=[{'m': m_in, 'n': n_in}],
            mode='total',
        )
        ok += res['verified']

print(f'gcd verified on {ok} (m, n) pairs in [1, 24]² (no counter-examples)')

gcd verified on 576 (m, n) pairs in [1, 24]² (no counter-examples)


### §4.7.3 Quadratic equation solver

```
x := 0; y := 0;
while ¬(y = 1) do (
    if a×x×x + b×x + c = 0 then y := 1
    else (if a×x×x − b×x + c = 0 then y := 1 else skip);
    x := x + 1
)
```

**Why total correctness fails in general:** the loop searches `x = 0, 1, 2, …` for a solution. If no integer solution exists (e.g. `x² − 2x + 2 = 0`), the loop runs forever. So we can only prove **partial** correctness without an extra precondition.

**Partial spec:** `{a = ā ∧ b = b̄ ∧ c = c̄} S {āx² + b̄x + c̄ = 0 ∨ ā(−x)² + b̄(−x) + c̄ = 0}`.

Reading: **if** the program terminates, the value left in `x` (or its negation) is a solution. We claim nothing if it doesn't.

**Tweaks the chapter makes for the proof:** the body has to be slightly restructured (move the `x := x + 1` and add a check at `x = 0`) so the invariant `y = 1 ↔ Q(x)` holds before the loop entry. See chapter §4.7.3.

**Variant for the conditional total version:** if there's a solution `k ∈ ℤ` with `|k| − x ≥ 0`, the variant is `|k| − x`. Each iteration increases `x`, so the variant decreases.

### §4.7.4 The Collatz conjecture — termination open

```
while ¬(n = 1) do (
    x := n; m := 0;
    while 2 ≤ x do (x := x − 1; m := m + 1);
    if x = 0 then n := m else n := 3 × n + 1
)
```

(The inner loop computes `m = ⌊n/2⌋` if `n` is even, leaves `n = 1` otherwise. Then the if-branch implements the Collatz step: `n/2` for even, `3n + 1` for odd.)

**The Collatz conjecture** says this terminates for every `n ∈ ℕ`. Verified for `n ≤ 2.95 × 10²⁰` by computer search; **unproved in general**.

**What we can say:**

- **Partial correctness:** `{n = n̄ ∈ ℕ} S {n̄ = 1}` — *if* it terminates, then `n̄ = 1` (because the only way to exit the outer loop is `n = 1`, and we never modify the ghost `n̄`). Wait — actually `n` *is* modified by the loop. The chapter writes the postcondition as `n̄ = 1`, which is odd because `n̄` is the *original* value. 

Reading the chapter charitably: it likely meant `n = 1` in the post, since that's what the loop guarantees on exit. **The interesting claim is just "if you exit, you exit with `n = 1`".**

- **Total correctness for bounded inputs:** `{n = n̄ ∈ ℕ ∧ n̄ ≤ 2.95 × 10²⁰} S {⇓ n = 1}` — established empirically.

- **Total correctness in general:** **unknown.** Proving this is the open problem.

## §3. Practical guidance (§4.8)

Three skills that take practice:

### Picking a good Hoare triple

**Problems with bad specs:**

1. **Too weak** — satisfied by trivial programs. Always check: does `m := 0; n := 1; ...` (or some similar reset) satisfy the spec? If yes, add ghost variables.
2. **Too strong** — implies things the program doesn't actually do. The proof gets stuck.
3. **Captures the wrong intent** — passes the proof but doesn't match what users want. Read the spec out loud as English.

### Finding loop invariants

From §4.8.2, three guidelines:

1. **Relate to the postcondition.** Often part of the post can be lifted into the invariant directly. (Division: `m = d·n + r` works as both invariant and core post.)
2. **Involve the variables that change.** Variables fixed by the loop usually don't need to appear.
3. **Look at concrete iterations.** Trace the loop on a small input. What relationship between the variables holds *every* iteration?

**Tip:** `verify_triple` with the candidate invariant as the postcondition is a fast sanity check. If it fails empirically, it's not an invariant.

### Finding loop variants

From §4.8.3:

1. **Look at the loop condition.** Whatever variable governs exit usually appears in the variant. (`while x ≤ n do …` → variant is something like `n − x`.)
2. **For "while ¬(switch = 1)" patterns** (where the body eventually sets a flag), variants are *much* harder. Often the variant must encode "how far we are from satisfying the exit condition," which can require knowing things about the inputs.
3. **Combine variables when single ones don't decrease monotonically.** GCD's `x + y` is the canonical example.

## §4. Proposition 13 — connection to operational semantics

From §4.9. The Hoare-logic system isn't a separate world — it lines up exactly with what we built in chapters 1–2.

**Proposition 13 (i)** (partial):

$$\{P\}\ S\ \{Q\} \quad \iff \quad \forall \sigma. P(\sigma) \implies \bigl(\exists \sigma'. \langle S, \sigma\rangle \Rightarrow^* \langle \textbf{skip}, \sigma'\rangle \implies Q(\sigma')\bigr)$$

Reading: **the partial Hoare triple holds iff for every state in P that *can reach* skip, the post holds at the resulting state.** Non-terminating runs are exempt.

**Proposition 13 (ii)** (total):

$$\{P\}\ S\ \{\Downarrow Q\} \quad \iff \quad \forall \sigma. P(\sigma) \implies \exists \sigma'.\, \langle S, \sigma\rangle \Downarrow \sigma' \,\land\, Q(\sigma')$$

Reading: **the total triple holds iff every state in P leads (via big-step) to some state in Q.** Termination is required.

**Why this matters:** the Hoare-logic rules are *sound* — every triple they prove is actually true under the operational semantics. They're also *complete* (in a precise mathematical sense) — true triples are derivable. The two systems agree.

We don't reproduce the proof of Prop 13. Sketch: induction on the structure of S, with each Hoare rule mirroring the corresponding small-step (or big-step) inference.

## Exercise 32 — Hoare triples for primality, φ, φ⁻¹

> For each program, state a precondition + postcondition that fully describes the intended behaviour, and a stronger postcondition for inputs that don't match.

### (a) Primality test (Example 4)

```
y := n − 1; z := 1;
while 2 ≤ y ∧ z = 1 do (
    r := n;
    while y ≤ r do r := r − y;
    if r = 0 then z := 0 else skip;
    y := y − 1
)
```

**Spec:**

$$\{n = \bar n \,\land\, \bar n \ge 2\}\ S\ \{\Downarrow z = 1 \iff \bar n \text{ is prime}\}$$

**Cross-check vs official solution:** the official spec uses `{n = n̄ ∈ ℕ}` (no `n̄ ≥ 2` constraint). My stricter precondition is more accurate: for `n̄ ∈ {0, 1}` the outer loop never fires (since `2 ≤ n̄ − 1` is false), so the program returns `z = 1`. But neither 0 nor 1 is prime by convention, so the iff `z = 1 ↔ n̄ prime` would technically *fail* in those cases. Either spec is defensible (the official may charitably exclude the n̄ ≥ 2 boundary by convention); mine is sharper.

**Loop invariant for the outer loop:** `z = 0 ↔ ∃k ∈ ℕ. y < k ≤ n − 1 ∧ k divides n` (i.e. "z is 0 iff we've found a divisor in `(y, n−1]`").

### (b) and (c) φ and φ⁻¹ (Section 6.2.2)

These reference Section 6.2.2 (covered in N15). With the standard Cantor-style pairing function:

- For φ: `{m = m̄ ∈ ℕ ∧ n = n̄ ∈ ℕ} S {⇓ x = φ(m̄, n̄)}` — **matches official.**
- For φ⁻¹ (using only φ in pre/post): `{n = n̄ ∈ ℕ} S {⇓ φ(x, y) = n̄}` — input is single n; output is a pair (x, y) such that φ(x, y) = n̄.

(The official PDF for (c) appears to have a transcription mistake — its triple looks identical to (b)'s. My version is the standard reading: φ⁻¹ takes a *single* natural and outputs a *pair*, with the spec expressed via φ rather than φ⁻¹.)

## Exercise 33 — Hoare triples for `n mod m` and the Ex 61 programs

### (a) `n mod m` (from Exercise 5)

```
r := n;
while r ≤ -1 do r := r + m;
while m ≤ r do r := r - m
```

**Spec:**

$$\{n = \bar n \,\land\, m = \bar m > 0\}\ S\ \{\Downarrow r = \bar n \bmod \bar m\}$$

(Mathematically `n mod m` for `m > 0` is the unique `r ∈ [0, m)` with `n = q·m + r` for some `q ∈ ℤ`.)

**Two loop invariants:**

1. First loop (`while r ≤ -1`): invariant `n̄ ≡ r (mod m̄)` (i.e. `r` differs from `n̄` by a multiple of `m̄`). Variant: `−r` (decreases as `r` increases toward 0).
2. Second loop (`while m ≤ r`): same invariant `n̄ ≡ r (mod m̄)`. Variant: `r` (decreases as we subtract `m̄`).

**Final post:** the invariant plus `0 ≤ r < m̄` (from the loop exit conditions) is exactly what `r = n̄ mod m̄` means.

### (b) The two programs from Exercise 61 (now Ex 19 in our N6)

Both compute the triangular number `T(i) = i(i+1)/2`.

**Program left:**
```
y := 1; a := 0;
while y ≤ i do (a := a + y; y := y + 1)
```

**Spec:** `{i = ī ≥ 0} S {⇓ a = ī(ī+1)/2}`

**Loop invariant:** `a = (y−1)·y/2 ∧ 1 ≤ y ≤ ī + 1`. After the loop (`y > ī`), `y = ī + 1`, so `a = ī·(ī+1)/2` ✓.

**Loop variant:** `ī + 1 − y`.

**Program right:**
```
x := i × (i+1); y := 0;
while 0 ≤ x do (x := x − 2; y := y + 1)
```

**Spec:** `{i = ī ≥ 0} S {⇓ y = ī(ī+1)/2 + 1}` — the off-by-one shows up because the loop runs once after `x = 0` (still satisfying `0 ≤ x`).

Equivalent to left program iff we adjust by 1, which the chapter glosses over. Real equivalence holds modulo this.

**Loop invariant:** `x = ī(ī+1) − 2·y ∧ y ≥ 0`. 

**Loop variant:** `x + 2` (always ≥ 1 inside the loop, decreases by 2 each iter).

## Exercise 34 — `m^n` full derivation

> ```
> x := 1; y := n;
> while y > 1 do (x := x × m; y := y − 1)
> ```
> (a) Adjust the program if needed. (b) Pre/post. (c) Loop invariant. (d) Partial derivation. (e) Loop variant. (f) Total derivation.

### (a) Adjusting the program

**Bug:** the loop condition is `y > 1`, which means it stops at `y = 1` — but `m¹ = m` should require multiplying by `m` once. With `y > 1`: when `y` starts at `n`, the loop multiplies `n − 1` times. We end up with `m^(n − 1)`, not `m^n`. Off by one.

**Fix:** change to `y ≥ 1` (i.e. `1 ≤ y`):

```
x := 1; y := n;
while 1 ≤ y do (x := x × m; y := y − 1)
```

Now: starts with `x = 1, y = n`. After `n` iterations: `x = m^n`, `y = 0`. ✓

**Edge case:** `n = 0`. Loop body never runs, `x` stays at `1 = m^0`. ✓

### (b) Spec

$$\{m = \bar m \,\land\, n = \bar n \ge 0\}\ S\ \{\Downarrow x = \bar m^{\bar n}\}$$

### (c) Loop invariant

`I` = `x · m^y = m̄^n̄ ∧ y ≥ 0 ∧ m = m̄`.

**Why it's an invariant:** body does `x := x · m; y := y − 1`. After: `(x · m) · m^(y − 1) = x · m^y`. So `x · m^y` doesn't change. ✓

**Holds initially:** `x · m^y = 1 · m̄^n̄ = m̄^n̄`. ✓

**At loop exit (`y < 1` i.e. `y = 0`):** `x · m^0 = x = m̄^n̄`. ✓ → matches the post.

### (d) Partial derivation

```
1. {m = m̄ ∧ n = n̄ ≥ 0} x := 1; y := n {x = 1 ∧ y = n̄ ∧ m = m̄ ∧ n̄ ≥ 0}    by := × 2 and ;
   The post implies x · m^y = m̄^n̄ (since 1 · m̄^n̄ = m̄^n̄), and y ≥ 0.
2. {I ∧ 1 ≤ y} x := x × m; y := y − 1 {I}    by := and ;
   Need: (x · m) · m^(y − 1) = m̄^n̄, which simplifies to x · m^y = m̄^n̄ ∈ I. ✓
3. {I} while 1 ≤ y do (...) {I ∧ ¬(1 ≤ y)}    from 2 by while rule.
   The post `I ∧ y < 1 ∧ y ≥ 0` forces y = 0, so x · m^0 = x = m̄^n̄. ✓
4. {m = m̄ ∧ n = n̄ ≥ 0} S {x = m̄^n̄}    from 1, 3 by ; and consequence.
```

### (e) Loop variant: `y`

- `y ≥ 0` is in the invariant.
- Each body iteration: `y := y − 1`, strictly decreasing.

### (f) Total derivation

Identical to (d) but with `⇓` everywhere. The variant `y` does the work in step 3.

In [3]:
# Empirical check on the FIXED power program.
power_prog = '''
    x := 1; y := n;
    while 1 <= y do (
        x := x * m;
        y := y - 1
    )
'''

ok = 0
for m_in in range(0, 6):
    for n_in in range(0, 8):
        res = verify_triple(
            precond  = lambda s, m=m_in, n=n_in: s.get('m', 0) == m and s.get('n', 0) == n,
            prog     = power_prog,
            postcond = lambda s, m=m_in, n=n_in: s.get('x', 0) == m ** n,
            sample_states=[{'m': m_in, 'n': n_in}],
            mode='total',
        )
        ok += res['verified']

print(f'm^n verified for {ok} (m, n) pairs in [0, 5] × [0, 7] (no counter-examples)')

m^n verified for 48 (m, n) pairs in [0, 5] × [0, 7] (no counter-examples)

## Exercise 35 — sum of first n positive integers

> ```
> x := 0; i := 1;
> while i ≤ n do (x := x + i; i := i + 1)
> ```

### (a) Spec

$$\{n = \bar n \ge 0\}\ S\ \{\Downarrow x = \bar n(\bar n + 1)/2\}$$

**Cross-check vs official:** the official spec uses `{n = n̄ ∈ ℕ \ {0}}` (positive naturals only). Mine uses `n̄ ≥ 0` (non-negative). My version is **strictly more general** — for n̄ = 0 the loop never fires, leaving `x = 0`, which equals `0·1/2 = 0` (and the empty sum `Σᵢ₌₁⁰ i = 0`). Both specs are correct; mine just covers an extra valid case.

### (b) Loop invariant

`I = x = (i − 1)·i/2 ∧ 1 ≤ i ≤ n̄ + 1 ∧ n = n̄`.

**Argument:** before the body, `x = (i − 1)·i/2`. After `x := x + i`, `x = (i − 1)·i/2 + i = i·(i + 1)/2`. After `i := i + 1`, the invariant becomes `x = (i − 1)·i/2` with the *new* `i = old i + 1`, which is exactly `i·(i + 1)/2` from before. ✓

**Initial:** `x = 0 = (1 − 1)·1/2`, `i = 1`, fine.

**At loop exit (`i > n̄`):** `i = n̄ + 1`, so `x = ((n̄ + 1) − 1)·(n̄ + 1)/2 = n̄(n̄ + 1)/2`. ✓

### (c) Partial derivation

```
1. {n = n̄ ≥ 0} x := 0; i := 1 {x = 0 ∧ i = 1 ∧ n = n̄ ≥ 0}    by := × 2 and ;
   This implies I (since x = 0 = (1−1)·1/2 and 1 ≤ 1 ≤ n̄+1).
2. {I ∧ i ≤ n̄} x := x + i; i := i + 1 {I}    by := × 2 (substitution check shown above).
3. {I} while i ≤ n do (...) {I ∧ ¬(i ≤ n̄)}    from 2 by while rule.
4. {n = n̄ ≥ 0} S {x = n̄(n̄ + 1)/2}    from 1, 3 by ; and consequence.
```

### (d) Loop variant: `n − i + 1`

(Equivalently: `n + 1 − i`.) Always `≥ 0` while inside the loop (`i ≤ n` ⟹ `n − i + 1 ≥ 1 > 0`); each iteration `i := i + 1` decreases the variant by 1. **Matches official.**

### (e) Total derivation

Same as (c), with `⇓` everywhere.

In [4]:
sum_prog = '''
    x := 0; i := 1;
    while i <= n do (
        x := x + i;
        i := i + 1
    )
'''

ok = 0
for n_in in range(0, 30):
    expected = n_in * (n_in + 1) // 2
    res = verify_triple(
        precond  = lambda s, n=n_in: s.get('n', 0) == n,
        prog     = sum_prog,
        postcond = lambda s, e=expected: s.get('x', 0) == e,
        sample_states=[{'n': n_in}],
        mode='total',
    )
    ok += res['verified']

print(f'sum 1..n verified for n ∈ [0, 29] ({ok} cases)')

sum 1..n verified for n ∈ [0, 29] (30 cases)


## Exercise 36 — integer logarithm (base 2)

> ```
> x := 1; z := 0;
> while x × 2 ≤ n do (z := z + 1; x := x × 2)
> ```

### (a) What is the integer logarithm?

`log₂_⌊⌋(n) = ⌊log₂(n)⌋` — the greatest integer `k` such that `2^k ≤ n`. So:

- `log₂_⌊⌋(1) = 0` (since `2⁰ = 1 ≤ 1 < 2`)
- `log₂_⌊⌋(2) = 1` (since `2¹ = 2 ≤ 2 < 4`)
- `log₂_⌊⌋(7) = 2` (since `2² = 4 ≤ 7 < 8`)
- `log₂_⌊⌋(8) = 3`

Defined for `n ≥ 1` (we'd need `n > 0` for `log` to make sense).

### (b) Spec

$$\{n = \bar n \ge 1\}\ S\ \{\Downarrow z = \lfloor \log_2 \bar n \rfloor \,\land\, x = 2^z\}$$

### (c) Loop invariant

`I = x = 2^z ∧ x ≤ n̄ ∧ z ≥ 0 ∧ n = n̄`.

**Argument:** before the body, `x = 2^z` and `x ≤ n̄`. The body fires only if `x · 2 ≤ n̄`, i.e. `2^(z+1) ≤ n̄`. After `z := z + 1; x := x · 2`: `x = 2^(z+1) = 2^(new z)`, and `x ≤ n̄`. ✓

**Initial:** `x = 1 = 2⁰`, `z = 0`, `1 ≤ n̄` (from precondition), `z = 0 ≥ 0`. ✓

**At loop exit (`x · 2 > n̄`):** combined with `x ≤ n̄`: `x ≤ n̄ < 2x = 2^(z+1)`. So `2^z ≤ n̄ < 2^(z+1)`, which means `z = ⌊log₂ n̄⌋`. ✓

### (d) Partial derivation

```
1. {n = n̄ ≥ 1} x := 1; z := 0 {x = 1 = 2^0 ∧ z = 0 ∧ x ≤ n̄ ∧ n = n̄}    by := × 2 and ;
2. {I ∧ x · 2 ≤ n̄} z := z + 1; x := x × 2 {I}    by := × 2 (algebra check above).
3. {I} while x × 2 ≤ n do (...) {I ∧ ¬(x · 2 ≤ n̄)}    from 2 by while rule.
   Post implies z = ⌊log₂ n̄⌋ ∧ x = 2^z.
4. {n = n̄ ≥ 1} S {z = ⌊log₂ n̄⌋ ∧ x = 2^z}    from 1, 3 by ; and consequence.
```

### (e) Loop variant: `n − x`

Always `≥ 0` inside the loop (since `x ≤ n̄`). Each iteration: `x := x · 2`, so `n − x` decreases. Wait — does it strictly decrease? If `x = n̄`, then `x · 2 = 2n̄ > n̄`, so the loop exits. Inside the loop `x · 2 ≤ n̄`, i.e. `x ≤ n̄/2`, so `n̄ − x ≥ n̄/2 > 0`. After: `n̄ − 2x ≤ n̄ − x − x ≤ n̄ − x − 1` (since `x ≥ 1`). So strictly decreases. ✓

### (f) Total derivation

Same as (d) with `⇓`. Variant supplies termination.

In [5]:
log_prog = '''
    x := 1; z := 0;
    while x * 2 <= n do (
        z := z + 1;
        x := x * 2
    )
'''

ok = 0
for n_in in range(1, 100):
    expected_z = math.floor(math.log2(n_in))
    res = verify_triple(
        precond  = lambda s, n=n_in: s.get('n', 0) == n,
        prog     = log_prog,
        postcond = lambda s, e=expected_z: s.get('z', 0) == e and s.get('x', 0) == 2**e,
        sample_states=[{'n': n_in}],
        mode='total',
    )
    ok += res['verified']

print(f'integer log verified for n ∈ [1, 99] ({ok} cases)')

integer log verified for n ∈ [1, 99] (99 cases)


## Exercise 37 — divisibility-by-3 of the last digit

> Give a While program that tells whether the last digit of `n ∈ ℕ` is divisible by 3. Make it simple, not efficient.

### (a) Program

Strategy: extract the last decimal digit (`n mod 10`), then test divisibility by 3.

```
d := n;                              # get n mod 10 by subtracting 10s
while 10 ≤ d do d := d − 10;
                                     # now d is the last digit ∈ [0, 9]
r := d;                              # test if r mod 3 = 0 by subtracting 3s
while 3 ≤ r do r := r − 3;
                                     # now r ∈ {0, 1, 2}
if r = 0 then z := 1 else z := 0
```

### (b) Spec

$$\{n = \bar n \ge 0\}\ S\ \{\Downarrow z = 1 \iff (\bar n \bmod 10) \bmod 3 = 0\}$$

(Last digit divisible by 3 ⟺ last digit ∈ {0, 3, 6, 9}.)

### (c) Loop invariants + variants

**First loop:** invariant `d ≡ n̄ (mod 10) ∧ d ≥ 0`. Variant: `d`.
**Second loop:** invariant `r ≡ d (mod 3) ∧ r ≥ 0` where the original `d` is the last digit. Variant: `r`.

### (d), (e), (f) Derivations

Three building blocks chained with `;`:

```
1. {n = n̄ ≥ 0} d := n; while 10 ≤ d do (d := d − 10) {⇓ d = n̄ mod 10 ∧ 0 ≤ d < 10}
2. (continuing) r := d; while 3 ≤ r do (r := r − 3) {⇓ r = (n̄ mod 10) mod 3 ∧ 0 ≤ r < 3}
3. (continuing) if r = 0 then z := 1 else z := 0 {⇓ z = 1 ↔ (n̄ mod 10) mod 3 = 0}
```

Each piece is straightforward: while-rule with the invariant + variant; if-rule for the conditional. Total proof = chain via `;`.

Sanity check:

In [6]:
lastdigit_div3 = '''
    d := n;
    while 10 <= d do (d := d - 10);
    r := d;
    while 3 <= r do (r := r - 3);
    if r = 0 then
        z := 1
    else (
        z := 0
    )
'''

ok = 0
for n_in in range(0, 50):
    last = n_in % 10
    expected_z = 1 if last % 3 == 0 else 0
    res = verify_triple(
        precond  = lambda s, n=n_in: s.get('n', 0) == n,
        prog     = lastdigit_div3,
        postcond = lambda s, e=expected_z: s.get('z', 0) == e,
        sample_states=[{'n': n_in}],
        mode='total',
    )
    ok += res['verified']

print(f'last-digit-divisible-by-3 verified for n ∈ [0, 49] ({ok} cases)')

# Show a few examples
print()
for n_in in [10, 13, 26, 33, 47, 99]:
    final = run(lastdigit_div3, {'n': n_in})
    last = n_in % 10
    print(f'  n = {n_in:>2}, last digit = {last}, z = {final.get("z", 0)}  ({"divisible" if final.get("z", 0) == 1 else "not divisible"} by 3)')

last-digit-divisible-by-3 verified for n ∈ [0, 49] (50 cases)

  n = 10, last digit = 0, z = 1  (divisible by 3)
  n = 13, last digit = 3, z = 1  (divisible by 3)
  n = 26, last digit = 6, z = 1  (divisible by 3)
  n = 33, last digit = 3, z = 1  (divisible by 3)
  n = 47, last digit = 7, z = 0  (not divisible by 3)
  n = 99, last digit = 9, z = 1  (divisible by 3)


## Exercise 38 — Hoare rule for `repeat S until b`

> (a) Provide a partial-correctness rule. (b) Show that if `{P} repeat S until b {Q}_r` holds in your system, then for the equivalent While program from Exercise 3, `{P} S' {Q}` holds in the standard system.

### (a) The rule

Recall `repeat S until b` runs `S` *first*, then checks `b`; loops while `b` is false. So the body always runs at least once.

$$\dfrac{\{P\}\ S\ \{Q\} \qquad \{Q \,\land\, \neg\,\mathcal{B}\,b\}\ S\ \{Q\}}{\{P\}\ \textbf{repeat } S \textbf{ until } b\ \{Q \,\land\, \mathcal{B}\,b\}}_r$$

**Reading:**
- First premise: starting from `P`, one execution of `S` reaches `Q`.
- Second premise: from `Q ∧ ¬b`, another execution of `S` re-establishes `Q`. (`Q` is the loop invariant.)
- Conclusion: after the whole repeat-until, we're in `Q ∧ b` (we exited because `b` became true).

### (b) Connection to the equivalent While program

From Exercise 3, `repeat S until b` is equivalent to `S; while ¬b do S`. So `S' = S; while ¬b do S`.

Suppose `{P} repeat S until b {Q ∧ b}_r` holds in our new system. We want to show `{P} S; while ¬b do S {Q ∧ b}` in the standard system.

From our two premises:
1. `{P} S {Q}` — first execution.
2. `{Q ∧ ¬b} S {Q}` — body inside the standard while loop.

Apply the standard while rule to (2): `{Q} while ¬b do S {Q ∧ ¬¬b} = {Q ∧ b}`.

Apply the standard `;` rule to (1) and the result: `{P} S; while ¬b do S {Q ∧ b}` ✓.

**The two systems agree.** ∎

## Summary

**Total correctness** = partial correctness + termination. Notation: `{P} S {⇓ Q}`.

**The new while rule** has two extra ingredients beyond the partial version:
1. The invariant `P` implies the variant `a ≥ 0`.
2. The body strictly decreases `a` (proven via a fresh ghost `k = a` in pre, `a < k` in post of body).

**Loop variant** ≠ **loop invariant**:
- Invariant = predicate preserved by the body. Encodes what the loop *does*.
- Variant = non-negative arithmetic expression that strictly decreases. Proves the loop *terminates*.

**Common variant patterns:**
- `n − x` for counter loops bounded above by `n`.
- `x` for counter loops decreasing toward 0.
- `x + y` when neither alone decreases (gcd).
- `|k| − x` when the loop searches for a known target `k`.

**Across the four chapter examples** (β, gcd, quadratic, Collatz):
- β has no loops, total correctness is automatic.
- gcd: invariant about common-divisors; variant `x + y`.
- quadratic: only partial correctness in general (could loop forever); total only with extra precondition.
- Collatz: open problem — partial only; total requires unproven mathematical conjecture.

**Proposition 13** ties Hoare logic back to operational semantics: the formal systems agree on what's true.

**Next:** N11 quiz tests recognition of pre/post/invariant/variant patterns and rule application across all of chapter 4.